# Fase 5 — Análisis Exploratorio Profundo
**Input:** `data/clean/pizza_features.csv`  
**Output:** tablas agregadas en `data/clean/agg_*.csv`

Preguntas de negocio a responder (definidas en Fase 1):
1. ¿Cuánto dinero generamos por mes? ¿Hay estacionalidad?
2. ¿Cuáles son nuestros bestsellers? ¿Hay pizzas que eliminar?
3. ¿Cuántos clientes por día? ¿Hay horas pico?
4. ¿Cuál es el ticket promedio por pedido?
5. ¿Qué días de la semana vendemos más?

In [ ]:
import pandas as pd
import numpy as np

CLEAN = '../data/clean/'
df = pd.read_csv(CLEAN + 'pizza_features.csv', parse_dates=['date'])

day_order = ['Lunes','Martes','Miércoles','Jueves','Viernes','Sábado','Domingo']
print('Shape:', df.shape)

## KPIs Globales

In [ ]:
print(f'Revenue total 2015:     ${df["revenue"].sum():,.2f}')
print(f'Total pedidos:          {df["order_id"].nunique():,}')
print(f'Total pizzas vendidas:  {df["quantity"].sum():,}')
print(f'Ticket promedio:        ${df.groupby("order_id")["revenue"].sum().mean():.2f}')
print(f'Dias de operacion:      {df["date"].nunique()}  (el restaurante cerro 7 dias en 2015)')
print(f'Pedidos promedio/dia:   {df["order_id"].nunique() / df["date"].nunique():.1f}')

## P1 — Revenue mensual y estacionalidad
**Insight:** Julio es el mes pico ($72,557). Octubre el peor ($64,027).
La caída de Oct se explica parcialmente por 4 lunes consecutivos de cierre (5, 12, 19, 26 Oct).

In [ ]:
monthly = (df.groupby(['month', 'month_name', 'quarter'])
             .agg(revenue=('revenue','sum'), orders=('order_id','nunique'), pizzas=('quantity','sum'))
             .reset_index().sort_values('month'))
monthly['avg_order_value'] = (monthly['revenue'] / monthly['orders']).round(2)
print(monthly[['month_name','quarter','revenue','orders','avg_order_value']].to_string(index=False))
monthly.to_csv(CLEAN + 'agg_monthly.csv', index=False)

## P2 — Bestsellers y candidatos a eliminar
**Insight:** Top 3 son Chicken. La Classic Deluxe es la más vendida en unidades pero 5° en revenue.  
**Candidato claro a revisar:** Brie Carre — solo existe en S, $11,588 revenue, 490 unidades (el más bajo).

In [ ]:
pizza_perf = (df.groupby(['pizza_type_id','name','category'])
                .agg(revenue=('revenue','sum'), qty=('quantity','sum'), orders=('order_id','nunique'))
                .reset_index().sort_values('revenue', ascending=False))
pizza_perf['revenue_rank'] = pizza_perf['revenue'].rank(ascending=False).astype(int)
pizza_perf['revenue_pct']  = (pizza_perf['revenue'] / pizza_perf['revenue'].sum() * 100).round(2)
pizza_perf['abc_class']    = pd.cut(pizza_perf['revenue_rank'], bins=[0,8,20,32], labels=['A','B','C'])

print('TOP 10 (por revenue):')
print(pizza_perf.head(10)[['name','category','revenue','qty','abc_class']].to_string(index=False))
print('\nBOTTOM 5 (candidatos a revisar):')
print(pizza_perf.tail(5)[['name','category','revenue','qty','abc_class']].to_string(index=False))

print('\nRevenue por categoría:')
cat = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print((cat / cat.sum() * 100).round(1))

pizza_perf.to_csv(CLEAN + 'agg_pizza_performance.csv', index=False)

## P3 — Clientes por día y horas pico
**Insight:** Hora pico = 12:00h (2,520 pedidos). Noche (18-20) es el 2° bloque más activo.  
El restaurante necesita máximo personal a las 12h y a las 18-19h.

In [ ]:
hourly = (df.groupby('hour')
            .agg(orders=('order_id','nunique'), revenue=('revenue','sum'))
            .reset_index().sort_values('hour'))
print('Pedidos por hora:')
print(hourly.to_string(index=False))
hourly.to_csv(CLEAN + 'agg_hourly.csv', index=False)

block = (df.groupby('time_block', observed=True)
           .agg(orders=('order_id','nunique'), revenue=('revenue','sum'))
           .reset_index())
block['pct_orders'] = (block['orders'] / block['orders'].sum() * 100).round(1)
print('\nBloques horarios:')
print(block.to_string(index=False))

## P4 — Ticket promedio
**Insight:** Ticket promedio $38.31, mediana $32.50. La diferencia sugiere pedidos grandes
que jalan el promedio hacia arriba. El ticket máximo es $444.20 (pedido de grupo/evento).

In [ ]:
tickets = df.groupby(['order_id','date','day_name_es','month'])['revenue'].sum().reset_index()
tickets.columns = ['order_id','date','day_name_es','month','ticket']
tickets['n_pizzas'] = df.groupby('order_id')['quantity'].sum().values

print(f'Ticket promedio: ${tickets["ticket"].mean():.2f}')
print(f'Ticket mediana:  ${tickets["ticket"].median():.2f}')
print(f'Ticket max:      ${tickets["ticket"].max():.2f}')
print(f'Tickets > $50:   {(tickets["ticket"] > 50).sum()} ({(tickets["ticket"] > 50).mean()*100:.1f}%)')
tickets.to_csv(CLEAN + 'agg_tickets.csv', index=False)

## P5 — Días de la semana
**Insight:** Viernes es el rey (3,538 pedidos, $136K). Domingo es el más tranquilo (2,624 pedidos).  
El patrón Jue-Vie-Sáb concentra el 44% del volumen semanal.

In [ ]:
daily = (df.groupby(['day_of_week','day_name_es'])
           .agg(revenue=('revenue','sum'), orders=('order_id','nunique'), pizzas=('quantity','sum'))
           .reset_index().sort_values('day_of_week'))
daily['pct_orders'] = (daily['orders'] / daily['orders'].sum() * 100).round(1)
print(daily[['day_name_es','orders','revenue','pct_orders']].to_string(index=False))
daily.to_csv(CLEAN + 'agg_daily.csv', index=False)

## Extra — Talla por categoría y tendencia semanal

In [ ]:
cat_size = (df.groupby(['category','size_label','size_order'])
              .agg(revenue=('revenue','sum'), qty=('quantity','sum'))
              .reset_index().sort_values(['category','size_order']))
print('Talla por categoría:')
print(cat_size.to_string(index=False))
cat_size.to_csv(CLEAN + 'agg_category_size.csv', index=False)

weekly = (df.groupby('week')
            .agg(revenue=('revenue','sum'), orders=('order_id','nunique'))
            .reset_index().sort_values('week'))
weekly.to_csv(CLEAN + 'agg_weekly.csv', index=False)
print('\nTablas agg_*.csv generadas en data/clean/')

## Insights accionables (en lenguaje de negocio)

1. **Los viernes necesitan doble personal.** El viernes concentra un 34% más de pedidos que el domingo y un 27% más que el lunes. Si el staffing es parejo toda la semana, el restaurante está subservido los viernes y sobrestaffed los domingos.

2. **El turno de almuerzo (12-14h) y noche (18-20h) son el 61% de los pedidos.** El restaurante puede planificar turnos split: preparar ingredientes entre 10-11h y entre 16-17h para absorber los picos.

3. **Las pizzas Chicken generan el mayor revenue individual pero Classic domina en volumen.** Conviene mantener y promocionar Thai Chicken, BBQ Chicken y California Chicken — son las 3 pizzas de mayor revenue. Tienen margen mayor que las Classics más baratas.

4. **Brie Carre es candidata a retiro de menú.** Es la pizza de menor revenue ($11,588), solo existe en talla S, y tiene el menor número de unidades vendidas (490 en un año). Su espacio en el menú podría reemplazarse con una variante de alto desempeño.

5. **Octubre fue el peor mes — parcialmente por cierres.** El restaurante estuvo cerrado 4 lunes consecutivos en octubre (5, 12, 19, 26). Normalizado por días de operación, octubre no es tan diferente. Antes de concluir estacionalidad negativa en otoño, hay que validar si esos cierres fueron intencionales o incidentes.